# 04 · Validate — cross-variant breadth + head-to-head

**Standard slot:** *validate (in silico).* **For Project 07 the core science is BREADTH:** does each top
binder hold across a panel of variant RBDs? Report the **worst-case** variant, not the best (D3 pt 2).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Cross-variant breadth

Model each top candidate against the variant panel and record per-variant `pae_interaction`. A broad
neutralizer keeps **all** variants low. The panel below is **EXAMPLE_DATA** (mock); use your real
verified variant RBDs in the project.

In [ ]:
import binder_tools as bt, pandas as pd, numpy as np

ranked = pd.read_csv("results/ranked.csv")
top = ranked.head(10)
VARIANTS = ["Wuhan", "Delta", "Omicron_BA1", "Omicron_XBB", "SARS1"]   # EXAMPLE panel — verify/replace
rows = []
for r in top.itertuples():
    prof = bt.breadth_across_variants(str(r.sequence), VARIANTS, tool="mock")
    prof["design_id"] = r.design_id
    prof["worst_case_pae"] = max(v for v in prof.values() if isinstance(v, (int, float)))
    rows.append(prof)
breadth = pd.DataFrame(rows).set_index("design_id")
breadth = breadth.sort_values("worst_case_pae")   # broadest first (lowest worst-case)
breadth.to_csv("results/breadth.csv")
print("breadth profile (SYNTHETIC) — broadest (lowest worst-case pae) first:")
breadth

In [ ]:
import matplotlib.pyplot as plt
panel = [c for c in breadth.columns if c not in ("worst_case_pae",)]
fig, ax = plt.subplots(figsize=(7, 4))
for did, row in breadth.iterrows():
    ax.plot(panel, [row[c] for c in panel], marker="o", alpha=0.6, label=str(did)[:18])
ax.set_ylabel("pae_interaction (lower = better)"); ax.set_xlabel("variant RBD")
ax.set_title("Cross-variant breadth (EXAMPLE_DATA / mock)")
ax.axhline(10, ls="--", c="k", lw=0.8, label="binder cutoff 10")
plt.xticks(rotation=30); plt.tight_layout(); plt.savefig("results/proj07_breadth.png", dpi=150); plt.show()

## 2 · Head-to-head: BindCraft vs RFdiffusion `[core]`

Compare the two paradigms on survival and interface confidence (honest hit-rate accounting).

In [ ]:
designs = pd.read_csv("results/designs.csv")
by = designs.groupby("paradigm").agg(n=("design_id", "size"),
        mean_pae=("pae_interaction", "mean"), mean_sc=("shape_complementarity", "mean"))
print("per-paradigm summary (SYNTHETIC):")
print(by)
print("\n[reminder] all numbers are mock/EXAMPLE_DATA — compare paradigms on YOUR real campaign.")

## D3 (part 2) checklist
- [ ] Cross-variant breadth table + figure; **worst-case** variant reported per candidate.
- [ ] Conserved- vs variable-epitope comparison (breadth difference).
- [ ] BindCraft vs RFdiffusion head-to-head (hit rate, interface confidence, novelty).
- [ ] Failure-mode notes: which candidates are narrow/escape-prone, and why.

**Next:** `05_validation_plan.ipynb` — the controlled, biosafe validation + breadth-testing plan.